# leafmap: biblioteca para análisis geoespacial y mapas interactivos

<div style="display: flex; justify-content: flex-start;">
  <a href="https://colab.research.google.com/github/tpb708-programacionsig/2026-i/blob/main/contenidos/iv-procesamiento-datos-geoespaciales/09-leafmap.ipynb">
    <img src="https://img.shields.io/badge/Abrir%20en-Colab-F9AB00?logo=googlecolab&logoColor=white" alt="Abrir en Colab" style="margin: 0;">
  </a>
</div>

**NOTA IMPORTANTE**

Los mapas generados con leafmap están diseñados para ejecutarse en cuadernos Jupyter (en [Google Colab](https://colab.research.google.com/) o en un entorno local como [Visual Studio Code](https://code.visualstudio.com/)), por lo que **no se visualizan en páginas HTML estáticas** como las de este sitio. Por esa razón, en este capítulo los resultados se muestran mediante **capturas de pantalla**. Para ver los mapas de forma interactiva, ejecute el cuaderno en Colab o en un entorno local.

## Trabajo previo

### Lecturas

Tenkanen, H., Heikinheimo, V., & Whipp, D. (2024). *Introduction to Python for Geographic Data Analysis*. CRC Press. [https://pythongis.org/](https://pythongis.org/)
\
\
Lovelace, R., Nowosad, J., & Müller, J. (2024). *Geocomputation with Python*. CRC Press. [https://py.geocompx.org/](https://py.geocompx.org/)

### Otros recursos

Documentación oficial de leafmap\
[leafmap documentation](https://leafmap.org/)

  - [API Reference](https://leafmap.org/leafmap/)
  - [Tutorials](https://leafmap.org/tutorials/)

Repositorio y notebooks de ejemplo de leafmap\
[opengeos/leafmap (GitHub)](https://github.com/opengeos/leafmap)

Referencia de paletas de colores de matplotlib\
[Colormap reference — matplotlib documentation](https://matplotlib.org/stable/gallery/color/colormap_reference.html)

## Introducción

[leafmap](https://leafmap.org/) es una biblioteca de Python para crear visualizaciones interactivas de datos geoespaciales en cuadernos Jupyter. Ha sido desarrollada con base en otras bibliotecas de mapeo como [folium](https://python-visualization.github.io/folium), [ipyleaflet](https://ipyleaflet.readthedocs.io/), [maplibre](https://maplibre.org/), [bokeh](https://docs.bokeh.org/en/latest/docs/user_guide/topics/geo.html), [pydeck](https://deckgl.readthedocs.io/), [kepler.gl](https://docs.kepler.gl/docs/keplergl-jupyter) y [plotly](https://plotly.com/python/maps). leafmap proporciona una interfaz de programación de aplicaciones (API) unificada para acceder a las funcionalidades de estas bibliotecas.

A diferencia de folium, que está orientada a la generación de mapas web, leafmap está orientada al **análisis geoespacial interactivo** dentro de Jupyter, con métodos de alto nivel para agregar datos vectoriales y raster, esquemas de clasificación, comparación de capas y muchas otras operaciones con poco código.

En este capítulo se detallan y ejemplifican algunas de las principales capacidades de leafmap.

## Instalación

### En ambientes locales (ej. conda)

Se recomienda actualizar primero conda y mamba.

```bash
# Actualizar conda y mamba
conda update conda
conda update -c conda-forge mamba
```

leafmap puede instalarse con `pip`, `conda` o `mamba`, desde la línea de comandos del sistema operativo. Solo es necesario hacerlo de una forma. Se recomienda instalarla junto con las bibliotecas asociadas para el manejo de datos vectoriales (geopandas) y raster (rasterio, localtileserver), así como mapclassify para los esquemas de clasificación.

```bash
# Instalar leafmap y sus bibliotecas asociadas con pip
pip install leafmap geopandas rasterio localtileserver mapclassify

# Instalar leafmap y sus bibliotecas asociadas con conda
conda install -c conda-forge leafmap geopandas rasterio localtileserver mapclassify

# Instalar leafmap y sus bibliotecas asociadas con mamba
mamba install -c conda-forge leafmap geopandas rasterio localtileserver mapclassify
```

### En la nube (ej. Google Colab)

leafmap **no** viene preinstalado en Google Colab. Ejecute la siguiente celda **si trabaja en Colab**. Si trabaja en un entorno local ya configurado (ej. conda), puede **omitirla**.

In [ ]:
# Instalación en Google Colab (omitir en entornos locales ya configurados)
!pip install -U leafmap geopandas rasterio localtileserver mapclassify

## Carga de bibliotecas

In [ ]:
# Para mapas interactivos
import leafmap

# Para datos tabulares
import pandas as pd

# Para datos vectoriales
import geopandas as gpd

# Para datos raster
import rasterio

# Para álgebra lineal
import numpy as np

# Para habilitar los widgets interactivos en Google Colab.
# En entornos locales (ej. VS Code) esta importación no aplica y se omite.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

leafmap proporciona soporte para diversos sistemas de mapeo (*plotting backends*), incluyendo folium, ipyleaflet, maplibre, bokeh, pydeck, keplergl y plotly. El que se usa por defecto es ipyleaflet.

Si se desea cambiar el sistema de mapeo debe utilizarse una de las siguientes sentencias `import`.

```python
import leafmap.foliumap as leafmap
import leafmap.bokehmap as leafmap
import leafmap.maplibregl as leafmap
import leafmap.deck as leafmap
import leafmap.kepler as leafmap
import leafmap.plotlymap as leafmap
```

## Operaciones básicas

### Creación y configuración general de mapas

El constructor de la clase [`Map`](https://leafmap.org/leafmap/#leafmap.leafmap.Map) crea un mapa interactivo con un mapa base. Por defecto, se utiliza el mapa base de `OpenStreetMap`.

In [ ]:
# Crear un mapa leafmap
m = leafmap.Map()

# Desplegar el mapa
m

<figure style="text-align: center;">
  <img
    src="img/leafmap-basico.png"
    alt="Mapa leafmap básico"
  >
  <figcaption><strong>Figura 1</strong>. Mapa leafmap básico con el mapa base de OpenStreetMap.</figcaption>
</figure>

El mapa puede ser personalizado con argumentos como `center`, `zoom` y `height`.

In [ ]:
# Crear un mapa leafmap personalizado
m = leafmap.Map(
    center=(9.6, -84.2),
    zoom=7,
    height="400px"
)

# Desplegar el mapa
m

<figure style="text-align: center;">
  <img
    src="img/leafmap-personalizado.png"
    alt="Mapa leafmap personalizado centrado en Costa Rica"
  >
  <figcaption><strong>Figura 2</strong>. Mapa leafmap personalizado, centrado en Costa Rica.</figcaption>
</figure>

Por defecto, un mapa leafmap incluye controles para acercamiento/alejamiento, escala, atribución y barra de herramientas. Estos controles pueden activarse y desactivarse.

In [ ]:
# Activar y desactivar controles de un mapa leafmap
m = leafmap.Map(
    center=(9.6, -84.2),
    zoom=7,
    height="400px",
    zoom_control=True,
    draw_control=False,
    scale_control=True,
    fullscreen_control=False,
    attribution_control=False,
    toolbar_control=True
)

# Desplegar el mapa
m

<figure style="text-align: center;">
  <img
    src="img/leafmap-sincontroles.png"
    alt="Mapa leafmap con algunos controles desactivados"
  >
  <figcaption><strong>Figura 3</strong>. Mapa leafmap con algunos controles desactivados.</figcaption>
</figure>

### Control de búsqueda

El método [`add_search_control()`](https://leafmap.org/leafmap/#leafmap.leafmap.Map.add_search_control) agrega un control de búsqueda al mapa, el cual permite que los usuarios busquen lugares por sus nombres.

In [ ]:
# Dirección del servicio de búsqueda
url = "https://nominatim.openstreetmap.org/search?format=json&q={s}"

# Agregar un control de búsqueda
m.add_search_control(url, position="topleft")

# Desplegar el mapa
m

<figure style="text-align: center;">
  <img
    src="img/leafmap-busqueda.png"
    alt="Mapa leafmap con un control de búsqueda"
  >
  <figcaption><strong>Figura 4</strong>. Mapa leafmap con un control de búsqueda de lugares por nombre.</figcaption>
</figure>

## Manejo de datos vectoriales

### Mapa de puntos desde un archivo CSV

El método [`add_points_from_xy()`](https://leafmap.org/leafmap/#leafmap.leafmap.Map.add_points_from_xy) agrega a un mapa los puntos de un dataframe que contiene columnas de longitud y latitud. Es una forma directa de mapear datos como los registros de presencia de especies de [GBIF](https://www.gbif.org/).

En el siguiente ejemplo se cargan registros de presencia de [puma (*Puma concolor*)](https://es.wikipedia.org/wiki/Puma_concolor) en Costa Rica y se muestran como puntos.

In [ ]:
# Cargar los registros de presencia desde un archivo CSV
puma_df = pd.read_csv(
    'https://raw.githubusercontent.com/tpb708-programacionsig/2026-i/refs/heads/main/datos/gbif/gbif-puma-concolor-cri.csv'
)

# Crear un mapa leafmap
m = leafmap.Map(center=(9.9, -84.0), zoom=7, height="400px")

# Agregar una capa base
m.add_basemap("CartoDB.Positron")

# Agregar los puntos a partir de las columnas de longitud y latitud
m.add_points_from_xy(
    puma_df,
    x="decimalLongitude",
    y="decimalLatitude",
    layer_name="Puma concolor"
)

# Desplegar el mapa
m

> El mapa interactivo resultante se despliega al ejecutar el cuaderno en Colab o en un entorno local. Por tratarse de un mapa ipyleaflet, no se incluye aquí una captura estática.

### Agregar un geodataframe

El método [`add_gdf()`](https://leafmap.org/leafmap/#leafmap.leafmap.Map.add_gdf) agrega un geodataframe como una capa. En el siguiente ejemplo se cargan los polígonos de los países del mundo desde [Natural Earth](https://www.naturalearthdata.com/).

In [ ]:
# Crear un geodataframe con datos y polígonos de paises
paises_gdf = gpd.read_file(
    'https://github.com/tpb708-programacionsig/2026-i/raw/refs/heads/main/datos/natural-earth/paises.gpkg'
)

In [ ]:
# Crear mapa leafmap
m = leafmap.Map(height="400px")

# Agregar capa base
m.add_basemap("CartoDB.Positron")

# Definir estilo
style = {"color": "black", "fillColor": "black", "fillOpacity": 0.1, "weight": 2}

# Agregar un geodataframe al mapa
m.add_gdf(
    paises_gdf,
    style=style,
    layer_name="Países",
    zoom_to_layer=True
)

# Desplegar el mapa
m

<figure style="text-align: center;">
  <img
    src="img/leafmap-addgdf.png"
    alt="Mapa leafmap con los polígonos de países"
  >
  <figcaption><strong>Figura 5</strong>. Polígonos de países agregados a un mapa leafmap con add_gdf().</figcaption>
</figure>

### Mapas de coropletas

El método [`add_data()`](https://leafmap.org/leafmap/#leafmap.leafmap.Map.add_data) agrega datos vectoriales a un mapa y puede emplear varios esquemas de clasificación. En el siguiente bloque de código se utiliza para generar un mapa de coropletas según la variable de población.

In [ ]:
# Crear mapa leafmap
m = leafmap.Map(height="400px")

# Agregar capa de datos vectoriales
m.add_data(
    paises_gdf,
    column="POP_EST", # columna para el mapa de coropletas
    scheme="Quantiles", # esquema de clasificación
    cmap="Blues", # paleta de colores
    legend_title="Población estimada"
)

# Desplegar el mapa
m

<figure style="text-align: center;">
  <img
    src="img/leafmap-adddata.png"
    alt="Mapa de coropletas de población estimada por país"
  >
  <figcaption><strong>Figura 6</strong>. Mapa de coropletas de población estimada por país, generado con add_data().</figcaption>
</figure>

## Manejo de datos raster

El método [`add_raster()`](https://leafmap.org/leafmap/#leafmap.leafmap.Map.add_raster) agrega una capa raster al mapa. En el siguiente ejemplo se carga un raster de altitud de Costa Rica, proveniente del sitio [WorldClim](https://www.worldclim.org/).

In [ ]:
# Lectura de datos de altitud
altitud = rasterio.open(
    'https://github.com/tpb708-programacionsig/2026-i/raw/refs/heads/main/datos/worldclim/altitud.tif'
)

In [ ]:
# Crear un objeto Map de leafmap
m = leafmap.Map(
    height="400px",
    center=[altitud.bounds.bottom, altitud.bounds.left],
    zoom=7
)

# Agregar el raster al mapa
m.add_raster(
    'https://github.com/tpb708-programacionsig/2026-i/raw/refs/heads/main/datos/worldclim/altitud.tif',
    colormap='viridis',
    layer_name='Altitud'
)

# Desplegar el mapa
m

<figure style="text-align: center;">
  <img
    src="img/leafmap-raster.png"
    alt="Mapa leafmap con el raster de altitud de Costa Rica"
  >
  <figcaption><strong>Figura 7</strong>. Raster de altitud de Costa Rica agregado a un mapa leafmap con add_raster().</figcaption>
</figure>

Los bloques de código siguientes crean un mapa con dos imágenes de satélite [Sentinel-2](https://es.wikipedia.org/wiki/Sentinel-2) de la zona de [Gandoca-Manzanillo](https://es.wikipedia.org/wiki/Refugio_nacional_de_vida_silvestre_Gandoca-Manzanillo), en dos fechas diferentes. El método [`download_file()`](https://leafmap.org/leafmap/#leafmap.common.download_file) descarga los archivos.

In [ ]:
# URL de la imagen 1
url1 = "https://github.com/tpb708-programacionsig/2026-i/raw/refs/heads/main/datos/sentinel/gandoca-20240105.tif"
archivo1 = "gandoca-20240105.tif"

# URL de la imagen 2
url2 = "https://github.com/tpb708-programacionsig/2026-i/raw/refs/heads/main/datos/sentinel/gandoca-20240618.tif"
archivo2 = "gandoca-20240618.tif"

# Descargar las imágenes en archivos locales
leafmap.download_file(url1, archivo1, quiet=True, overwrite=True)
leafmap.download_file(url2, archivo2, quiet=True, overwrite=True)

In [ ]:
# Crear el mapa
m = leafmap.Map(zoom=20)

# Agregar una capa base
m.add_basemap("CartoDB.Positron")

# Agregar las capas raster con las imágenes
m.add_raster(archivo1, indexes=[4, 3, 2], opacity=0.7, layer_name="2024-01-05")
m.add_raster(archivo2, indexes=[4, 3, 2], opacity=0.7, layer_name="2024-06-18")

# Desplegar el mapa
m

<figure style="text-align: center;">
  <img
    src="img/leafmap-gandoca.png"
    alt="Mapa leafmap con dos imágenes Sentinel-2 de Gandoca"
  >
  <figcaption><strong>Figura 8</strong>. Dos imágenes de satélite Sentinel-2 de Gandoca superpuestas en un mapa leafmap.</figcaption>
</figure>

El método [`split_map()`](https://leafmap.org/leafmap/#leafmap.leafmap.Map.split_map) crea un mapa que permite comparar dos capas, mediante un control deslizante.

In [ ]:
# Crear el mapa
m = leafmap.Map(
    center=[9.633083, -82.676167],
    zoom=18
)

# Agregar una capa base
m.add_basemap("CartoDB.Positron")

# Agregar el mapa dividido
m.split_map(
    archivo1,
    archivo2,
    left_label="2024-01-05",
    right_label="2024-06-18",
    left_args={"bands": [4, 3, 2], "opacity": 0.7},
    right_args={"bands": [4, 3, 2], "opacity": 0.7},
)

# Desplegar el mapa
m

<figure style="text-align: center;">
  <img
    src="img/leafmap-gandocasplit.png"
    alt="Mapa dividido que compara dos imágenes Sentinel-2 de Gandoca"
  >
  <figcaption><strong>Figura 9</strong>. Comparación de dos imágenes Sentinel-2 de Gandoca con un mapa dividido (split_map).</figcaption>
</figure>

## Ejercicios

1. A partir del conjunto de datos que ha utilizado en sus tareas, elabore un mapa de leafmap que muestre los registros que contengan coordenadas (latitud y longitud) con el método `add_points_from_xy()`.
2. Elabore un mapa de coropletas con `add_data()` que muestre la variable de [tasa de mortalidad infantil](https://datos.bancomundial.org/indicador/SP.DYN.IMRT.IN) por país. Una los polígonos de países de Natural Earth (`https://github.com/tpb708-programacionsig/2026-i/raw/refs/heads/main/datos/natural-earth/paises.gpkg`) con el indicador del Banco Mundial (`https://raw.githubusercontent.com/tpb708-programacionsig/2026-i/refs/heads/main/datos/world-bank/paises-tasa-mortalidad-infantil.csv`), utilizando los datos del año 2022.
3. Despliegue en un mapa dividido (`split_map()`) las capas raster correspondientes al [Índice de Vegetación de Diferencia Normalizada (NDVI)](https://es.wikipedia.org/wiki/%C3%8Dndice_de_vegetaci%C3%B3n_de_diferencia_normalizada) de las dos imágenes de satélite mostradas en este capítulo y observe las diferencias.
4. Busque imágenes de dos tiempos diferentes de otra zona afectada por deforestación o avance de la frontera agrícola y observe las diferencias en un mapa dividido. Sugerencias para la selección de la zona: Sierpe, Caño Negro.